# Training Utilities: SGD, Warmup + Cosine LR, Gradient Clipping, EMA

**Difficulty**: 🟡 Medium

### Problem Statement

Every training script has the same scaffolding around `loss.backward()`, whether it trains a ResNet, a diffusion U-Net or an LLM:

- an **optimizer step**, here SGD with momentum, Nesterov momentum and weight decay;
- a **learning-rate schedule**. Linear warmup followed by cosine decay is the default in most modern recipes: MAE and CLIP use it, and LLaMA decays to 10% of the peak LR after 2000 warmup steps;
- **gradient clipping** by global norm. `clip_grad_norm_(params, 1.0)` appears in nearly every transformer recipe;
- an **exponential moving average (EMA)** of the weights. DDPM and Stable Diffusion sample from EMA weights, and the MoCo key encoder and the DINO/BYOL teachers are EMAs of the online network.

MLE and research-engineer interviews often ask you to write these from memory and then probe the details. Where does weight decay go? What does PyTorch store in the momentum buffer? What does `clip_grad_norm_` return? What happens to BatchNorm buffers in an EMA? Your versions must match PyTorch **exactly**: the grader runs them side by side with `torch.optim.SGD` and `torch.nn.utils.clip_grad_norm_`.

---

### Requirements

1. **`SGD(params, lr, momentum=0.0, weight_decay=0.0, nesterov=False)`** with `step()` and `zero_grad()`. Its updates must equal `torch.optim.SGD` (with `dampening=0`). For each parameter $\theta$ with gradient $\nabla$ at step $t = 1, 2, \dots$:

   $$g_t = \nabla + \lambda\,\theta_{t-1} \qquad \text{(weight decay = L2 penalty, added to the gradient)}$$
   $$b_1 = g_1, \qquad b_t = \mu\, b_{t-1} + g_t \quad (t > 1) \qquad \text{(momentum buffer, only if } \mu > 0)$$
   $$d_t = \begin{cases} g_t + \mu\, b_t & \text{nesterov} \\ b_t & \text{otherwise} \end{cases} \qquad (d_t = g_t \text{ when } \mu = 0), \qquad \theta_t = \theta_{t-1} - \eta\, d_t$$

   - `params` may be a **generator** such as `model.parameters()`.
   - Skip parameters whose `.grad` is `None` completely: no update, no weight decay, no buffer.
   - Keep one momentum buffer per parameter. Do not modify `p.grad` inside `step()`, and do not let the buffer share memory with `p.grad`.
   - The update must not be recorded by autograd.
   - Subclassing `torch.optim.Optimizer` is recommended: it gives you `param_groups`, a per-parameter `self.state` dict and `zero_grad()` for free. A plain class with `step()` and `zero_grad()` is fine too (store the learning rate as `self.lr`).
   - `nesterov=True` with `momentum=0` should raise `ValueError`, as PyTorch does (not graded).

2. **`warmup_cosine_lr(step, total_steps, warmup_steps, base_lr, min_lr=0.0) -> float`**, with a 0-indexed step $s$. Write $W$ = `warmup_steps`, $T$ = `total_steps`, $\eta_{\max}$ = `base_lr`, $\eta_{\min}$ = `min_lr`:

   $$\eta(s) = \begin{cases}
   \eta_{\max}\,\dfrac{s+1}{W} & s < W \\
   \eta_{\min} + \tfrac12(\eta_{\max}-\eta_{\min})\bigl(1+\cos(\pi p)\bigr), \quad p = \dfrac{s-W}{T-W} & W \le s < T \\
   \eta_{\min} & s \ge T
   \end{cases}$$

   - **Convention:** step 0 already uses `base_lr / warmup_steps`, not 0, so no step is wasted at LR 0. The last warmup step reaches `base_lr`. (Hugging Face's `get_cosine_schedule_with_warmup` uses `step / warmup` instead and starts at exactly 0.)
   - `warmup_steps = 0` means no warmup, so `lr(0) = base_lr`. Past `total_steps` the LR stays at `min_lr`.

3. **`clip_grad_norm_(params, max_norm) -> Tensor`** must match `torch.nn.utils.clip_grad_norm_`:
   - $\|g\| = \sqrt{\sum_{p}\sum_i g_{p,i}^2}$ is **one global L2 norm** over the gradients of all parameters together. Parameters with `grad=None` are ignored.
   - `clip_coef = max_norm / (total_norm + 1e-6)`. If `clip_coef < 1`, multiply **every** gradient in place by it. Otherwise leave the gradients untouched (clip, don't normalise).
   - Return the total norm **before** clipping (a 0-dim tensor; a float is accepted). If no parameter has a gradient, return 0. All-zero gradients must not turn into NaN.
   - `params` may be a generator.

4. **`EMA(model, decay)`** with `update(model)` and `copy_to(model)`:
   - The constructor takes a snapshot: shadow = a copy of the current parameters and buffers.
   - `update(model)`: for every parameter, $\text{shadow} \leftarrow \text{decay}\cdot\text{shadow} + (1-\text{decay})\cdot\theta$. **Buffers** (e.g. BatchNorm running stats, including the integer `num_batches_tracked`) are **copied**, not averaged.
   - `copy_to(model)` writes the shadow parameters and buffers into `model` **in place**. The model's parameters must remain the same trainable `nn.Parameter` objects.
   - The shadow must not share memory with the model and must not carry an autograd graph.
   - After $k$ updates with weights $\theta_1, \dots, \theta_k$ (and $\theta_0$ at construction): $\text{shadow}_k = \text{decay}^k\,\theta_0 + (1-\text{decay})\sum_{i=1}^{k}\text{decay}^{\,k-i}\,\theta_i$.

---

### Constraints

- ✅ Plain PyTorch tensor ops. You may subclass `torch.optim.Optimizer`.
- ❌ Do not call `torch.optim.SGD` (or `torch.optim._functional`), `torch.nn.utils.clip_grad_norm_`, `torch.optim.lr_scheduler`, or `torch.optim.swa_utils`. The test cells use them only as oracles.
- ✅ Everything runs on CPU in milliseconds.

---

<details>
  <summary>💡 Hint</summary>

  - Write the update rule out before coding. PyTorch's momentum buffer has **no** `(1 - momentum)` factor, and the learning rate multiplies the buffer only at the very end.
  - Weight decay is folded into the gradient *before* the momentum buffer sees it. What does that mean for a parameter whose `.grad` is `None`?
  - `model.parameters()` can be iterated only once. Both your optimizer and your clipping function iterate their params more than once.
  - Optimizer updates and EMA updates are not part of the model's computation graph. Which context manager keeps autograd out of them?
  - For the schedule, sketch lr(step) on paper and mark `warmup_steps` and `total_steps`. At which step does the cosine start, and what is its phase there?

</details>

---

In [ ]:
import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# A tiny network used by the tests. `unused` never takes part in forward(), so its
# .grad stays None: optimizers and clipping must skip it.
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(5, 8)
        self.fc2 = nn.Linear(8, 3)
        self.unused = nn.Parameter(torch.randn(4))

    def forward(self, x):
        return self.fc2(torch.tanh(self.fc1(x)))


def make_net(seed=0):
    torch.manual_seed(seed)
    return TinyNet()


print("parameters:", [(n, tuple(p.shape)) for n, p in make_net().named_parameters()])

In [ ]:
class SGD(torch.optim.Optimizer):
    """SGD with momentum / Nesterov / L2 weight decay, matching torch.optim.SGD (dampening=0).

    Args:
        params: iterable of Tensors (may be a one-shot generator such as model.parameters()).
        lr (float): learning rate eta.
        momentum (float): mu; 0 disables the momentum buffer.
        weight_decay (float): lambda; adds lambda * p to the gradient.
        nesterov (bool): use d = g + mu * buf instead of d = buf.
    """

    def __init__(self, params, lr, momentum=0.0, weight_decay=0.0, nesterov=False):
        # TODO: validate arguments (nesterov needs momentum > 0), then call
        #       super().__init__(params, defaults) with a dict of the hyper-parameters
        raise NotImplementedError

    def step(self, closure=None):
        # TODO: for each group / parameter with a gradient:
        #   g = grad (+ weight decay); update the momentum buffer kept in self.state[p];
        #   pick the direction (Nesterov or not); p <- p - lr * d   -- all without autograd
        raise NotImplementedError

In [ ]:
def sgd_trajectory(opt_cls, steps=20, **kw):
    net = make_net()
    opt = opt_cls(net.parameters(), lr=0.05, **kw)      # a generator, like real code
    g = torch.Generator().manual_seed(1)
    for _ in range(steps):
        x, y = torch.randn(6, 5, generator=g), torch.randn(6, 3, generator=g)
        opt.zero_grad()
        F.mse_loss(net(x), y).backward()
        opt.step()
    return dict((n, p.detach().clone()) for n, p in net.named_parameters())


for kw in [dict(), dict(momentum=0.9), dict(weight_decay=0.1),
           dict(momentum=0.9, weight_decay=0.1), dict(momentum=0.9, nesterov=True),
           dict(momentum=0.9, weight_decay=0.1, nesterov=True)]:
    mine, ref = sgd_trajectory(SGD, **kw), sgd_trajectory(torch.optim.SGD, **kw)
    for n in ref:
        assert torch.allclose(mine[n], ref[n], atol=1e-5), \
            f"SGD({kw}) diverges from torch.optim.SGD on {n} after 20 steps"
    assert torch.equal(mine["unused"], make_net().unused), "a parameter without .grad must not move"
print("SGD matches torch.optim.SGD for every option combination")

In [ ]:
def warmup_cosine_lr(step, total_steps, warmup_steps, base_lr, min_lr=0.0):
    """Linear warmup (step 0 -> base_lr / warmup_steps) then cosine decay to min_lr.

    Args:
        step (int): 0-indexed optimizer step.
        total_steps (int): step at which the cosine reaches min_lr (clamped afterwards).
        warmup_steps (int): number of linear-warmup steps (0 = no warmup).
        base_lr (float): peak learning rate.
        min_lr (float): final learning rate.

    Returns:
        float: the learning rate for this step.
    """
    # TODO: three regimes -- warmup, cosine, after the end
    raise NotImplementedError

In [ ]:
T, W, BASE, LO = 110, 10, 0.4, 0.01
lr = lambda s: float(warmup_cosine_lr(s, T, W, BASE, LO))
assert math.isclose(lr(0), BASE / W), f"step 0 should give base_lr/warmup = {BASE / W}, got {lr(0)}"
assert math.isclose(lr(W - 1), BASE), "the last warmup step reaches base_lr"
assert math.isclose(lr(W), BASE), "the cosine starts at base_lr at step == warmup_steps"
assert math.isclose(lr(W + (T - W) // 2), (BASE + LO) / 2), "cosine midpoint = (base + min) / 2"
assert math.isclose(lr(T), LO) and math.isclose(lr(T + 100), LO), "clamped to min_lr at/after the end"
sched = [lr(s) for s in range(T + 20)]
assert all(b > a for a, b in zip(sched[:W - 1], sched[1:W])), "strictly increasing warmup"
assert all(b <= a for a, b in zip(sched[W:], sched[W + 1:])), "non-increasing after warmup"
assert math.isclose(warmup_cosine_lr(0, 50, 0, 1.0), 1.0), "no warmup -> start at base_lr"
print("lr every 10 steps:", " ".join(f"{v:.3f}" for v in sched[::10]))

In [ ]:
def clip_grad_norm_(params, max_norm):
    """Clip gradients in place by their global L2 norm (torch.nn.utils.clip_grad_norm_ semantics).

    Args:
        params: iterable of Tensors (may be a generator); those with grad None are ignored.
        max_norm (float): maximum allowed global norm.

    Returns:
        Tensor: 0-dim total norm of all gradients BEFORE clipping (0 if there are none).
    """
    # TODO: collect the existing grads ONCE, compute one global norm,
    #       scale all grads in place by max_norm / (norm + 1e-6) when that is < 1
    raise NotImplementedError

In [ ]:
def net_with_grads(scale, seed=0):
    net = make_net(seed)
    for n, p in net.named_parameters():
        p.grad = None if n == "unused" else torch.randn_like(p) * scale
    return net

for scale, max_norm in [(3.0, 1.0), (0.01, 5.0)]:          # one clipped case, one untouched case
    mine, ref = net_with_grads(scale), net_with_grads(scale)
    got = clip_grad_norm_(mine.parameters(), max_norm)        # generator on purpose
    want = torch.nn.utils.clip_grad_norm_(ref.parameters(), max_norm)
    assert math.isclose(float(got), float(want), rel_tol=1e-5), f"norm {float(got)} != {float(want)}"
    for p, q in zip(mine.parameters(), ref.parameters()):
        assert (p.grad is None) == (q.grad is None)
        if q.grad is not None:
            assert torch.allclose(p.grad, q.grad, atol=1e-7), "clipped grads differ from PyTorch"

zero = make_net()
for p in zero.parameters():
    p.grad = torch.zeros_like(p)
assert float(clip_grad_norm_(zero.parameters(), 1.0)) == 0.0
assert all(torch.isfinite(p.grad).all() for p in zero.parameters()), "zero grads must not become NaN"
print("clip_grad_norm_ matches torch.nn.utils.clip_grad_norm_")

In [ ]:
class EMA:
    """Exponential moving average of a model's weights.

    Args:
        model (nn.Module): model whose current parameters/buffers initialise the shadow copy.
        decay (float): e.g. 0.999; shadow <- decay * shadow + (1 - decay) * param.
    """

    def __init__(self, model, decay):
        # TODO: store decay and detached CLONES of the parameters and buffers
        raise NotImplementedError

    def update(self, model):
        # TODO: average every parameter into the shadow (no autograd!); copy buffers
        raise NotImplementedError

    def copy_to(self, model):
        # TODO: write shadow parameters and buffers into model in place
        raise NotImplementedError

In [ ]:
torch.manual_seed(0)
bn_net = nn.Sequential(nn.Linear(4, 6), nn.BatchNorm1d(6), nn.ReLU(), nn.Linear(6, 2))
decay = 0.9
expected = {n: p.detach().clone() for n, p in bn_net.named_parameters()}   # theta_0
ema = EMA(bn_net, decay)
for k in range(5):
    with torch.no_grad():
        for p in bn_net.parameters():
            p.add_(torch.randn_like(p))       # pretend an optimizer step happened
    bn_net(torch.randn(8, 4) * 2 + 1)         # train-mode forward moves BN running stats
    ema.update(bn_net)
    for n, p in bn_net.named_parameters():
        expected[n] = decay * expected[n] + (1 - decay) * p.detach()

target = copy.deepcopy(bn_net)
ema.copy_to(target)
for n, p in target.named_parameters():
    assert torch.allclose(p, expected[n], atol=1e-6), f"EMA of {n} is not the closed form"
    assert isinstance(p, nn.Parameter) and p.requires_grad and p.grad_fn is None
for (n, b), b_live in zip(target.named_buffers(), bn_net.buffers()):
    assert torch.equal(b, b_live), f"buffer {n} must be copied from the model, not averaged"
print("EMA matches the closed form; buffers copied")

In [ ]:
# Putting it together: forward -> backward -> clip -> step -> schedule -> EMA, once per step.
def set_lr(optimizer, lr):
    for group in getattr(optimizer, "param_groups", []):   # torch.optim.Optimizer subclass
        group["lr"] = lr
    if hasattr(optimizer, "lr"):                             # plain class storing self.lr
        optimizer.lr = lr


def train_demo(min_lr, total=400, warmup=40, base_lr=0.1, verbose=False):
    torch.manual_seed(0)
    X = torch.randn(512, 5)
    Y = torch.sin(X @ torch.randn(5, 3)) + 0.05 * torch.randn(512, 3)
    model = nn.Sequential(nn.Linear(5, 64), nn.Tanh(), nn.Linear(64, 3))
    opt = SGD(model.parameters(), lr=0.0, momentum=0.9, weight_decay=1e-4, nesterov=True)
    ema = EMA(model, decay=0.99)                      # averages over ~1/(1-0.99) = 100 steps
    for step in range(total):
        lr = warmup_cosine_lr(step, total, warmup, base_lr, min_lr)
        set_lr(opt, lr)
        idx = torch.randint(0, 512, (32,))
        opt.zero_grad()
        loss = F.mse_loss(model(X[idx]), Y[idx])
        loss.backward()
        norm = clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()
        ema.update(model)
        if verbose and (step % 100 == 0 or step == total - 1):
            print(f"  step {step:3d}  lr {lr:.4f}  loss {loss.item():.4f}  grad-norm {float(norm):.3f}")
    ema_model = copy.deepcopy(model)
    ema.copy_to(ema_model)
    with torch.no_grad():
        return F.mse_loss(model(X), Y).item(), F.mse_loss(ema_model(X), Y).item()


for name, min_lr in [("cosine to 1e-3", 1e-3), ("warmup then constant", 0.1)]:
    online, averaged = train_demo(min_lr, verbose=(min_lr == 1e-3))
    print(f"{name:>22}:  full-data MSE online {online:.4f} | EMA {averaged:.4f}")
# Takeaway: when the LR is annealed to ~0 the iterates already settle and the EMA adds nothing
# (here it is even slightly worse: its window still contains weights from higher-LR steps).
# When the LR stays high (diffusion models often train at a constant LR), averaging the
# noisy iterates gives a clearly better model.

### Exercise (not graded): gradient accumulation

When a batch of 64 does not fit in memory, you run 4 micro-batches of 16, call `backward()` on each **without** zeroing the gradients in between, and step once. `.grad` *accumulates* (`+=`) across `backward()` calls. Implement `accumulated_grads` so the accumulated gradient equals the big-batch gradient exactly. How should each micro-batch loss be scaled, given that `nn.CrossEntropyLoss` averages over its batch?

Then run the BatchNorm cell below it. The equality breaks there. Why? And what else would break it (unequal micro-batch sizes, token-level losses with padding, dropout)?

In [ ]:
def big_batch_grads(model, X, y, loss_fn):
    model.zero_grad()
    loss_fn(model(X), y).backward()
    return [p.grad.clone() for p in model.parameters()]


def accumulated_grads(model, X, y, loss_fn, n_micro=4):
    """Same gradient as one big batch, computed from n_micro equal-size micro-batches."""
    model.zero_grad()
    # TODO: loop over zip(X.chunk(n_micro), y.chunk(n_micro)); scale each micro-batch loss so
    #       that the accumulated .grad equals the big-batch gradient; backward() each one.
    raise NotImplementedError
    return [p.grad.clone() for p in model.parameters()]

In [ ]:
torch.manual_seed(0)
X_acc, y_acc = torch.randn(64, 10), torch.randint(0, 3, (64,))
ce = nn.CrossEntropyLoss()
mlp = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
for a, b in zip(big_batch_grads(mlp, X_acc, y_acc, ce), accumulated_grads(mlp, X_acc, y_acc, ce)):
    assert torch.allclose(a, b, atol=1e-6), "accumulated gradient != big-batch gradient"
print("MLP: 4 micro-batches with loss/4 == one batch of 64")

bn_mlp = nn.Sequential(nn.Linear(10, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Linear(32, 3))
diff = max((a - b).abs().max().item() for a, b in zip(big_batch_grads(bn_mlp, X_acc, y_acc, ce),
                                                      accumulated_grads(bn_mlp, X_acc, y_acc, ce)))
print(f"with BatchNorm: max |grad difference| = {diff:.3e}   <- why is this not ~0?")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("training-utilities", SGD=SGD, warmup_cosine_lr=warmup_cosine_lr,
      clip_grad_norm_=clip_grad_norm_, EMA=EMA)